# 문제해결: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |
| 출처 확인 | 답변 근거가 되는 페이지와 청크를 확인할 수 있다. |
| 문제해결 | 검색 실패, 근거 부족, 청크 크기 문제를 점검할 수 있다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [ ]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas langchain_pymupdf4llm

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [1]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/Users/soohwi/sk-encoa/llm-workspace/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/qz/hyvszw9n4ms9wd1s5wn7fm2c0000gn/T/ipykernel_30454/4097930351.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 2. 환경변수 설정

OpenAI API Key는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-...
```

In [2]:
load_dotenv(override=True, dotenv_path="../.env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 OPENAI_API_KEY를 설정하세요."
    )

print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [3]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

In [4]:
loader = PyMuPDF4LLMLoader(str(PDF_PATH))
pages = loader.load()

# 메타데이터 보강
for doc in pages:
    doc.metadata["source_name"] = PDF_NAME
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1

print("로드된 페이지 수:", len(pages))
print("첫 번째 페이지 메타데이터:")
pprint(pages[0].metadata)
print("\n첫 번째 페이지 내용 일부:")
print(pages[0].page_content[:500])

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.

=== Document parser messages ===
                                                            Using Tesseract for OCR processing.
OCR on page.number=1/2.

=== Document parser messages ===
                                                                                                                        Using Tesseract for OCR processing.
OCR on page.number=2/3.

=== Document parser messages ===
                                                                                                                                                                                    Using Tesseract for OCR processing.
OCR on page.number=3/4.

=== Document parser messages ===
                                                                                                                                                                                                                                       

## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(pages)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

print("전체 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])

전체 청크 수: 20
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationDate': "D:20260529090329+09'00'",
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'file_path': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20260529090329+09'00'",
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'subject': '',
 'title': '',
 'total_pages': 15,
 'trapped': ''}

첫 번째 청크 내용:
# OrLtKnowledge* 

F OFFLY owaegcizs


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [6]:
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
5,6,1
6,7,3
7,8,3
8,9,1
9,10,1


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,36,# OrLtKnowledge* F OFFLY owaegcizs
1,1,2,107,dtLtKnowledget <!-- Start of picture text ...
2,2,3,104,dtLtKnowledge™ <!-- Start of picture text ...
3,3,4,339,OrtKnowledgeKnowledge <!-- Start of pictur...
4,4,5,296,dtLtKnowledge™ <!-- Start of picture text ...
5,5,6,480,dtLiKnowledget <!-- Start of picture text ...
6,6,7,779,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...
7,7,7,790,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
8,8,7,69,자료 : 언론보도 종합 자료 : 언론보도 종합 Hana Bank 6 Ha...
9,9,8,726,"- 지속적인 입법 노력에도 불구 여전히 거래 사기, 로맨스스캠 등 신종 사기 수법을..."


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print("FAISS 벡터스토어 생성 완료")
print("저장된 문서 청크 수:", vectorstore.index.ntotal)

FAISS 벡터스토어 생성 완료
저장된 문서 청크 수: 20


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [8]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=4)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.9779 | page=7 | chunk=6
# III. 금융사기 대응을 위한 노력   1. 규제 공백 해소를 위한 제도 보완      - 신규 사기 수법의 등장으로 발생하는 규제 공백을 해소하고자 법을 제정하거나 기존 법을 개정하여 책임 소재 기준, 피해 보상 체계 등을 지속적으로 정비         - ’24.1월 자본시장법을 개정하여 불공정거래행위에 대한 과징금을 신설하고, 부당이득에 관한 산정 기준을 명확히 하여 법적 실효성을 제고         - 3대 불공정거래행위(시세조종, 미공개정보 이용행위, 부정거래)에 대해 부당이득의 최대 2배까지 과징금을 부과하여 기존 형사처벌에 한정되어 있던 제재 수준을 확대         - 위반행위로 얻은 이익인 부당이득 산정 방식을 법제화하여 부당이득 금액 산정에 대한 기준을 제시함으로써 부당이득 조정으로 인한 처벌 수위 하향 문제를 해소         - ’24년 가상자산이용자보호법을 제정하여 가상자산 시세조종, 미공개 정보이용, 사기적 거래행위에 대한 처벌 기준을 마련하여 가상자

[2] score=1.0412 | page=7 | chunk=7
- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을 차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려         - 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간 경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입   ## 표3 | 전기통신금융사기 주요 대응 현황   ## 표4 | 투자사기 주요 대응 현황   |시기|주요 내용|시기||주요 내용| |---|---|---|---|---| |’23.7|통합신고대응센터 개소|’|자본시장법|Ÿ<br>불공정거래 부당이득의 최대 2배| |’24.1|은행권 자율배상제도 시행|24.1|개정|과징금 항목 신설| |’24.8|여신거래 안심차단 서비스 시행|’|가상자산이용자|

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=7, chunk=7
- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을 차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려         - 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간 경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입   ## 표3 | 전기통신금융사기 주요 대응 현황   ## 표4 | 투자사기 주요 대응 현황   |시기|주요 내용|시기||주요 내용| |---|---|---|---|---| 
--------------------------------------------------------------------------------
page=7, chunk=6
# III. 금융사기 대응을 위한 노력   1. 규제 공백 해소를 위한 제도 보완      - 신규 사기 수법의 등장으로 발생하는 규제 공백을 해소하고자 법을 제정하거나 기존 법을 개정하여 책임 소재 기준, 피해 보상 체계 등을 지속적으로 정비         - ’24.1월 자본시장법을 개정하여 불공정거래행위에 대한 과징금을 신설하고, 부당이득에 관한 산정 기준을 명확히 하여 법적 실효성을 제고         - 3대 불공정거래행위(시세조종, 미공개정보 이용행위, 부정거래)에 대해 부당이득의 최대 2배까지 과징금을 부과하여 기존 형
--------------------------------------------------------------------------------
page=12, chunk=15
- 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 기관과 대응책을 공유하는 방식으로 공동 대응      - 호주은행협회(ABA<sup>2)</sup> )와 고객소유은행협회(COBA<sup>3)</sup> )는 금융사기에 대응하기 위해 ’23년 Scam-Safe Accord 협약을 맺고 호주 소매은행의 공동 협의체

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [10]:
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),
    ("human", "{question}"),
])

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

answer_chain = prompt | llm | StrOutputParser()

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [11]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [12]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 주로 전화나 메시지를 통해 이루어지는 사기로, 보이스피싱과 같은 형태로 소비자를 속여 금전을 이체하게 만드는 방식입니다. 반면, 투자사기는 주식이나 가상자산 등 투자와 관련된 사기로, 미공개 정보 이용이나 시세 조종 등을 통해 피해자를 유인하여 부당한 이득을 취하는 것입니다. 두 가지 모두 금융사기지만, 발생하는 경로와 수법이 다릅니다. 근거: p.7, p.12

[검색된 근거]


,rank,page,chunk_id,preview
0,1,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
1,2,12,16,- 금융사기 수법 고도화에 따라 금융회사도 각종 탐지 및 예방 기술을 도입하여 금융...
2,3,12,15,- 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 기관과 대응책...
3,4,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...


In [13]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 금융사기를 예방하기 위해 다양한 탐지 및 예방 시스템을 도입하고 있습니다. 예를 들어, JPMorgan Chase는 AI 기반의 'NeuroShield' 시스템을 통해 실시간으로 금융사기를 탐지하고, Commonwealth Bank는 통화 여부에 기반한 'Scam Indicator' 프로그램을 운영하여 이체 시 사기 가능성을 판단합니다. 또한, 금융소비자가 이체 과정에서 이상 거래를 탐지할 수 있는 'NameCheck' 기술을 도입하여 보이스피싱 예방률을 높이고 있습니다. 근거: p.12

[검색된 근거]


,rank,page,chunk_id,preview
0,1,12,16,- 금융사기 수법 고도화에 따라 금융회사도 각종 탐지 및 예방 기술을 도입하여 금융...
1,2,12,15,- 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 기관과 대응책...
2,3,8,10,"- 해당 플랫폼에 참여하는 약 130여개의 금융회사는 총 9개 유형, 90개 항목으..."
3,4,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [14]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에서 확인할 수 없습니다.

[검색된 근거]


,rank,page,chunk_id,preview
0,1,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...
1,2,8,10,"- 해당 플랫폼에 참여하는 약 130여개의 금융회사는 총 9개 유형, 90개 항목으..."
2,3,8,9,"- 지속적인 입법 노력에도 불구 여전히 거래 사기, 로맨스스캠 등 신종 사기 수법을..."
3,4,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."


## 14. FAISS 인덱스 저장과 재로딩

실제 서비스에서는 매번 PDF를 읽고 임베딩하면 비효율적이다.  
한 번 만든 FAISS 인덱스를 로컬에 저장한 뒤 다시 불러올 수 있다.

주의: `allow_dangerous_deserialization=True`는 신뢰할 수 있는 로컬 파일을 불러올 때만 사용한다.

In [15]:
INDEX_DIR = Path("faiss_financial_fraud_index")

vectorstore.save_local(str(INDEX_DIR))
print("FAISS 인덱스 저장 완료:", INDEX_DIR.resolve())

loaded_vectorstore = FAISS.load_local(
    folder_path=str(INDEX_DIR),
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

print("FAISS 인덱스 재로딩 완료")
print("재로딩된 문서 청크 수:", loaded_vectorstore.index.ntotal)

FAISS 인덱스 저장 완료: /Users/soohwi/sk-encoa/llm-workspace/실습문제/faiss_financial_fraud_index
FAISS 인덱스 재로딩 완료
재로딩된 문서 청크 수: 20


## 15. 간단 검색 평가

정교한 RAG 평가는 별도 설계가 필요하지만, 수업에서는 우선 **Context Hit** 방식으로 검색 결과를 점검할 수 있다.

여기서는 기대 키워드가 검색된 상위 문서 안에 포함되어 있는지 확인한다.

In [16]:
eval_questions = [
    {
        "question": "금융사기는 일반적으로 어떤 두 가지로 구분되나요?",
        "expected_keywords": ["전기통신금융사기", "투자사기"],
    },
    {
        "question": "ASAP 플랫폼은 어떤 정보를 공유하나요?",
        "expected_keywords": ["보이스피싱", "정보공유", "AI플랫폼"],
    },
    {
        "question": "은행권 자율배상 제도는 무엇인가요?",
        "expected_keywords": ["자율배상", "보이스피싱", "개인정보"],
    },
    {
        "question": "해외에서는 금융사기에 어떻게 대응하고 있나요?",
        "expected_keywords": ["호주", "영국", "일본"],
    },
]

rows = []

for item in eval_questions:
    docs = vectorstore.similarity_search(item["question"], k=4)
    context = "\n".join(doc.page_content for doc in docs)
    hit_keywords = [kw for kw in item["expected_keywords"] if kw in context]
    rows.append({
        "question": item["question"],
        "expected_keywords": ", ".join(item["expected_keywords"]),
        "hit_keywords": ", ".join(hit_keywords),
        "hit_count": len(hit_keywords),
        "top_pages": sorted({doc.metadata.get("page_label") for doc in docs}),
    })

eval_df = pd.DataFrame(rows)
display(eval_df)

,question,expected_keywords,hit_keywords,hit_count,top_pages
0,금융사기는 일반적으로 어떤 두 가지로 구분되나요?,"전기통신금융사기, 투자사기","전기통신금융사기, 투자사기",2,"[7, 12]"
1,ASAP 플랫폼은 어떤 정보를 공유하나요?,"보이스피싱, 정보공유, AI플랫폼","보이스피싱, 정보공유, AI플랫폼",3,"[7, 8]"
2,은행권 자율배상 제도는 무엇인가요?,"자율배상, 보이스피싱, 개인정보","자율배상, 보이스피싱, 개인정보",3,"[7, 8]"
3,해외에서는 금융사기에 어떻게 대응하고 있나요?,"호주, 영국, 일본","호주, 영국",2,"[7, 12]"


## 16. 문제해결 실습 과제

아래 과제는 학습자가 직접 수정하며 해결하도록 설계한다.

### 과제 1. Top-k 변경 실험

같은 질문에 대해 `k=2`, `k=4`, `k=6`으로 바꾸어 검색 결과와 답변 품질을 비교한다.

확인할 것:

- k가 너무 작으면 필요한 근거가 빠지는가?
- k가 너무 크면 불필요한 문서가 섞이는가?
- 답변이 더 정확해지는가, 오히려 장황해지는가?

### 과제 2. Chunk Size 변경 실험

`chunk_size=400`, `800`, `1200`으로 바꾸어 다시 인덱스를 생성한다.

확인할 것:

- 청크가 너무 작으면 문맥이 끊기는가?
- 청크가 너무 크면 검색 정밀도가 떨어지는가?
- 표나 그림 설명이 어느 정도 보존되는가?

### 과제 3. 문서에 없는 질문 방어

문서에 없는 질문을 3개 만들어 넣어본다.

확인할 것:

- LLM이 문서 밖 지식을 섞어 답변하는가?
- “문서에서 확인할 수 없습니다.”라고 답하는가?
- 프롬프트를 어떻게 수정하면 방어력이 좋아지는가?

In [17]:
# 과제 1 실습 셀
# TODO: 질문과 k 값을 바꾸어가며 실행해보세요.

question = "무과실 배상책임제는 무엇이며 어떤 논의가 있나요?"

for k in [2, 4, 6]:
    print("=" * 100)
    print(f"k={k}")
    answer, sources = answer_question(question, k=k)
    print("\n[답변]")
    print(answer)
    print("\n[근거]")
    display(sources)

k=2

[답변]
문서에서 무과실 배상책임제에 대한 구체적인 내용이나 논의는 확인할 수 없습니다. 따라서 관련 정보는 제공할 수 없습니다. 문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...
1,2,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...


k=4

[답변]
문서에서 무과실 배상책임제에 대한 구체적인 내용이나 논의는 확인할 수 없습니다. 따라서 관련 정보는 제공할 수 없습니다. 문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...
1,2,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...
2,3,8,9,"- 지속적인 입법 노력에도 불구 여전히 거래 사기, 로맨스스캠 등 신종 사기 수법을..."
3,4,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."


k=6

[답변]
문서에서 무과실 배상책임제에 대한 구체적인 내용이나 논의는 확인할 수 없습니다. 따라서 관련 정보는 제공할 수 없습니다. 문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...
1,2,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...
2,3,8,9,"- 지속적인 입법 노력에도 불구 여전히 거래 사기, 로맨스스캠 등 신종 사기 수법을..."
3,4,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
4,5,12,15,- 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 기관과 대응책...
5,6,1,0,# OrLtKnowledge* F OFFLY owaegcizs


In [18]:
# 과제 2 실습 셀
# TODO: chunk_size를 바꿔가며 검색 결과를 비교해보세요.

# 예시 절차
# 1. 아래 chunk_size 값을 400, 800, 1200으로 변경
# 2. chunks_test 생성
# 3. vectorstore_test 생성
# 4. 같은 질문으로 similarity_search 결과 비교

chunk_size = 400
chunk_overlap = 80

test_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""],
)

chunks_test = test_splitter.split_documents(pages)

for i, doc in enumerate(chunks_test):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

vectorstore_test = FAISS.from_documents(
    documents=chunks_test,
    embedding=embeddings,
)

question = "ASAP 플랫폼은 어떤 역할을 하나요?"
results = vectorstore_test.similarity_search_with_score(question, k=4)

print(f"chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
print("청크 수:", len(chunks_test))

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))

chunk_size=400, chunk_overlap=80
청크 수: 31

[1] score=1.0935 | page=8 | chunk=14
- 24시간 신고대응센터를 운영하여 보이스피싱에 대응하고, 국내외 유관 기관과 업무적 협력을 통해 제도 허점 보완 및 예방 활동을 추진      - ’25.10월 금융위와 금감원이 공동으로 ‘보이스피싱 정보공유·분석 AI플랫폼 (ASAP)’을 출범하여 정부, 경찰, 금융회사 등 기관간 정보 공유 네트워크를 강화      - 금융·통신·수사 과정에서 파악된 보이스피싱 의심정보 등을 참여 기관이 실시간으로 공유하고, AI를 활용하여 패턴을 분석하는 등 범죄를 차단하기 위한 플랫폼 구축      - 해당 플랫폼에 참여하는 약 130여개의 금융회사는 총 9개 유형, 90개 항목으로 선별된 정보를 실시간으로 공유하고 범죄 예방에 활용

[2] score=1.3135 | page=8 | chunk=16
|표5 |‘보이스피|싱 근절 대책’ 3대 전략|표6 |ASAP 공유 정보 유형|| |---|---|---|---| |전략|주요 내용|유형<br>|항목| |거버넌스|Ÿ<br>유관기관간 통합 대응 조직 구축|1. 보이스피싱 피해를 입은 피해자의 계좌 관련 정보|14| |개편|Ÿ<br>수사·검거 등 기관 역량 강화|2. 보이스피싱 범죄에 이용된 계좌 관련 정보|18| ||Ÿ<br>악성앱 차단, 피싱 전화번호 긴급차단,|3. 보이스피싱 범죄 피해자 또는 범죄에 이용되었거나<br>이와 연계된 것으로 의심되는 계좌 관련 정보|15| |예방중심·<br>선제대응|<br>대포폰 개통 방지, 보이스피싱 AI 플랫폼<br>|4. 보이스피싱 범죄에 활용된 것으로 확인된 해외계좌 정보|8|

[3] score=1.3709 | page=7 | chunk=11
|’25.9|전기통신금융사기 통합대응단 출범|.||·<br>이익보장 약정 금지| |’25.10|보이스피싱 정보공유/분석 AI 플랫폼(ASAP) 출범|’25.3|국외 미신고 가상|자산사업자 앱 차단| |

In [19]:
# 과제 3 실습 셀
# TODO: 문서에 없는 질문을 넣어보고 답변 방어 여부를 확인하세요.

out_of_doc_questions = [
    "하나은행의 2026년 전체 금융사기 예방 예산은 얼마인가요?",
    "국내 모든 은행의 FDS 탐지 정확도 순위를 알려주세요.",
    "2027년에 시행될 금융사기 관련 법률 개정안의 최종 내용은 무엇인가요?",
]

for question in out_of_doc_questions:
    print("=" * 100)
    answer, sources = answer_question(question, k=4)
    print("[질문]")
    print(question)
    print("\n[답변]")
    print(answer)
    print("\n[근거]")
    display(sources)

[질문]
하나은행의 2026년 전체 금융사기 예방 예산은 얼마인가요?

[답변]
문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
1,2,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...
2,3,12,15,- 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 기관과 대응책...
3,4,8,10,"- 해당 플랫폼에 참여하는 약 130여개의 금융회사는 총 9개 유형, 90개 항목으..."


[질문]
국내 모든 은행의 FDS 탐지 정확도 순위를 알려주세요.

[답변]
문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
1,2,12,16,- 금융사기 수법 고도화에 따라 금융회사도 각종 탐지 및 예방 기술을 도입하여 금융...
2,3,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...
3,4,7,8,자료 : 언론보도 종합 자료 : 언론보도 종합 Hana Bank 6 Ha...


[질문]
2027년에 시행될 금융사기 관련 법률 개정안의 최종 내용은 무엇인가요?

[답변]
문서에서 확인할 수 없습니다.

[근거]


,rank,page,chunk_id,preview
0,1,7,7,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
1,2,7,6,# III. 금융사기 대응을 위한 노력 1. 규제 공백 해소를 위한 제도 보완 ...
2,3,8,10,"- 해당 플랫폼에 참여하는 약 130여개의 금융회사는 총 9개 유형, 90개 항목으..."
3,4,8,11,- 금융보안원은 금융사기 관련 정보를 지속적으로 축적하여 위험지표 산출 모델을 개발...


## 17. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
→ 출처 페이지 확인
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |